# 02 — Feature Engineering
### Global Airports ML Project
---
**Goal:** Transform raw airport data into rich features ready for ML models.

## 1. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')

## 2. Load & Clean Data

In [ ]:
from src.preprocessing import load_data, clean_data

df_raw   = load_data('data/airports.csv')
df_clean = clean_data(df_raw)
print(f'Clean shape: {df_clean.shape}')
df_clean.head(3)

## 3. Create New Features

In [ ]:
df = df_clean.copy()

df['altitude_category'] = pd.cut(
    df['altitude_ft'],
    bins=[-1, 500, 2000, 5000, 99999],
    labels=['Low', 'Medium', 'High', 'Very High']
)

df['passenger_tier'] = pd.cut(
    df['passengers_millions'],
    bins=[0, 20, 50, 80, 9999],
    labels=['Small', 'Medium', 'Large', 'Mega']
)

df['northern_hemisphere'] = (df['latitude']  >= 0).astype(int)
df['eastern_hemisphere']  = (df['longitude'] >= 0).astype(int)

df['passengers_per_runway'] = (df['passengers_millions'] / df['runways'].replace(0, 1)).round(2)

def extract_region(tz):
    if 'America'   in str(tz): return 'Americas'
    if 'Europe'    in str(tz): return 'Europe'
    if 'Asia'      in str(tz): return 'Asia'
    if 'Africa'    in str(tz): return 'Africa'
    if 'Australia' in str(tz): return 'Oceania'
    return 'Other'

df['region'] = df['timezone'].apply(extract_region)

print(f'New shape: {df.shape}')
print('New columns:', [c for c in df.columns if c not in df_clean.columns])

## 4. Visualise New Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

df['altitude_category'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0,0], color='coral', edgecolor='black')
axes[0,0].set_title('Altitude Category')
axes[0,0].tick_params(axis='x', rotation=0)

df['passenger_tier'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0,1], color='steelblue', edgecolor='black')
axes[0,1].set_title('Passenger Tier')
axes[0,1].tick_params(axis='x', rotation=0)

df['region'].value_counts().plot(
    kind='bar', ax=axes[1,0], color='green', alpha=0.8, edgecolor='black')
axes[1,0].set_title('Airport Region')
axes[1,0].tick_params(axis='x', rotation=30)

axes[1,1].hist(df['passengers_per_runway'], bins=12, color='purple', alpha=0.8, edgecolor='black')
axes[1,1].set_title('Passengers per Runway')
axes[1,1].set_xlabel('Passengers/Runway (millions)')

plt.suptitle('Engineered Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Label Encoding

In [ ]:
le = LabelEncoder()
cat_cols = df.select_dtypes(include=['object', 'category']).columns

df_encoded = df.copy()
for col in cat_cols:
    df_encoded[col + '_enc'] = le.fit_transform(df_encoded[col].astype(str))

print('Encoded columns:', [c for c in df_encoded.columns if c.endswith('_enc')])
df_encoded[['hub_type', 'hub_type_enc', 'region', 'region_enc']].head(6)

## 6. Feature Scaling

In [ ]:
feature_cols = [
    'latitude', 'longitude', 'altitude_ft',
    'passengers_millions', 'runways', 'passengers_per_runway'
]

X = df[feature_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

df_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

print('Before scaling:')
print(X.describe().round(2))
print('\nAfter scaling (mean≈0, std≈1):')
print(df_scaled.describe().round(2))

## 7. Correlation After Feature Engineering

In [ ]:
new_num_cols = feature_cols + ['northern_hemisphere', 'eastern_hemisphere']
corr_new = df[new_num_cols].corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr_new, dtype=bool))
sns.heatmap(corr_new, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation — After Feature Engineering', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Export Processed Data

In [ ]:
import os
os.makedirs('data', exist_ok=True)
df.to_csv('data/airports_processed.csv', index=False)
print('Saved → data/airports_processed.csv ✔')
df.head(3)

---
## Summary
| Feature | Description |
|---|---|
| `altitude_category` | Binned altitude (Low/Medium/High/Very High) |
| `passenger_tier` | Binned passengers (Small/Medium/Large/Mega) |
| `northern_hemisphere` | 1 if latitude ≥ 0 |
| `eastern_hemisphere` | 1 if longitude ≥ 0 |
| `passengers_per_runway` | Efficiency metric |
| `region` | Continent derived from timezone |

➡ Proceed to `03_clustering.ipynb`